In [13]:
import pandas as pd
import numpy as np
import os #for processing 20 files at once
def preprocess_and_generate_signals(folder_path):
    if not os.path.exists(folder_path):
        print(f"Error: The folder '{folder_path}' was not found.")
        return None

    summary_report = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            
            try:
                # 1. Load and sort
                df = pd.read_csv(file_path)
                # Validation: check if columns exist
                if 'Date' not in df.columns or 'Price' not in df.columns:
                    print(f"Skipping {filename}: Missing required columns.")
                    continue
                
                df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
                df = df.sort_values('Date').reset_index(drop=True)
                
                # 2. Cleaning
                df['Price'] = df['Price'].astype(str).str.replace(',', '').astype(float)
                
                # 3. Feature Engineering
                df['SMA_5'] = df['Price'].rolling(window=5).mean()
                df['SMA_20'] = df['Price'].rolling(window=20).mean()
                
                # 4. Signal Generation
                df['Recommendation'] = np.where(df['SMA_5'] > df['SMA_20'], 'BUY', 'WAIT')
                
                # 5. Alignment
                df['Target'] = df['Price'].shift(-1)
                
                # 6. Housekeeping (Drop NaNs)
                df = df.dropna()
                
                if df.empty:
                    print(f"Skipping {filename}: Insufficient data after processing.")
                    continue
                output_name = f"processed_{filename}"
                df.to_csv(output_name, index=False)
                
                summary_report.append({
                    'Company': filename.replace(' Historical Data.csv', ''),
                    'Signal_Date': df['Date'].iloc[-1].strftime('%d-%m-%Y'), # The date of the signal
                    'Last_Price': df['Price'].iloc[-1],
                    'Signal': df['Recommendation'].iloc[-1]
                })
                
            except Exception as e:
                print(f"Failed to process {filename}: {e}")
            
    return pd.DataFrame(summary_report)

my_data_folder = r'CSV' 
summary = preprocess_and_generate_signals(my_data_folder)
print(summary)

Skipping NIFTY 50-04-03-2025-to-04-03-2026.csv: Missing required columns.
                         Company Signal_Date  Last_Price Signal
0                           ADEL  02-03-2026     2124.60   WAIT
1                           AXBK  02-03-2026     1372.30    BUY
2                           BJFN  02-03-2026      978.25    BUY
3                           BRTI  02-03-2026     1873.20   WAIT
4                           HCLT  02-03-2026     1371.00   WAIT
5                           HDBK  02-03-2026      879.40   WAIT
6                            HLL  02-03-2026     2320.60   WAIT
7   ICBK Historical Data (1).csv  02-03-2026     1374.00   WAIT
8                           INFY  02-03-2026     1288.90   WAIT
9                            ITC  02-03-2026      314.90   WAIT
10                          MAHM  02-03-2026     3334.30   WAIT
11                          MRTI  02-03-2026    14388.00   WAIT
12                          NEST  02-03-2026     1279.70    BUY
13                          NT